Current State - ResNet50 [batch,512], DistilBERT[batch,768]
Now we concatenate so the model sees both Image and texts simultaneously(multimodal)

In [1]:
#load weights from previous files
import torch
import torch.nn as nn
from transformers import DistilBertTokenizer, DistilBertModel
import numpy as np
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

In [2]:
#loading image embedder
class ImageEmbedder(nn.Module):
  def __init__(self):
    super().__init__()

    resnet = models.resnet50(pretrained = True)
    for param in resnet.parameters():
      param.requires_grad=False
    self.backbone = nn.Sequential(*list(resnet.children())[:-1])
    self.embedding_head = nn.Sequential(
        nn.Flatten(),
        nn.Linear(2048,512),
        nn.ReLU()
    )

  def forward(self,x):
    return self.embedding_head(self.backbone(x))




In [3]:
#loading text embedder
class TextEmbedder(nn.Module):
  def __init__(self):
    super().__init__()

    self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
    for param in self.bert.parameters():
      param.requires_grad = False

    self.embedding_head = nn.Sequential(
        nn.Linear(768,768),
        nn.ReLU(),
        nn.Dropout(0.1)
    )

  def forward(self,input_ids, attention_mask):
    outputs = self.bert(input_ids = input_ids, attention_mask = attention_mask)
    cls_token = outputs.last_hidden_state[:,0,:]
    return self.embedding_head(cls_token)




In [4]:
#loading both embedders
image_embedder = ImageEmbedder()
text_embedder = TextEmbedder()

image_embedder.eval()
text_embedder.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 114MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TextEmbedder(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in

In [5]:
#building the fusion model
class MultimodalAdScorer(nn.Module):
  def __init__(self):
    super().__init__()

    #both embedders live in the model
    self.image_embedder = image_embedder
    self.text_embedder = text_embedder

    #fusion layer (text - 768) + (image - 512) = 1280 -> 512 -> 256 -> 1

    self.fusion = nn.Sequential(
      nn.Linear(1280,512),
      nn.ReLU(),
      nn.Dropout(0.3),
      nn.Linear(512,256),
      nn.ReLU(),
      nn.Dropout(0.3),
      nn.Linear(256, 1)
    )

  def forward(self, image_tensor, input_ids, attention_mask):
    img_emb = self.image_embedder(image_tensor)
    text_emb = self.text_embedder(input_ids, attention_mask)

    #concat both branches
    combined = torch.cat([img_emb, text_emb], dim = 1)

    score = self.fusion(combined) #running the fusion layer to produce a score

model = MultimodalAdScorer()
print(model)


MultimodalAdScorer(
  (image_embedder): ImageEmbedder(
    (backbone): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU

In [8]:
#building synthetic data because the dataset I am using doesnt have any image URLs
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

N = 200
syn_images = torch.randn(N, 3, 224, 224)

sample_texts = [
    "Buy now and save big today.",
    "Limited offer on premium products",
    "Fast delivery guaranteed worldwide",
    "Join millions of happy customers",
    "Exclusive deal ends at midnight"
]

all_texts = [sample_texts[i%len(sample_texts)] for i in range(N)] #generate 200 samples by cycling through 5 sample slogans

tokens = tokenizer(
    all_texts,
    return_tensors = 'pt',
    max_length = 128,
    padding = 'max_length',
    truncation = True
)

syn_score = torch.randn(N, 1)
print(f"Images: {syn_images.shape}")
print(f"Input_Ids: {tokens['input_ids'].shape}")
print(f"Scores: {syn_score.shape}")








Images: torch.Size([200, 3, 224, 224])
Input_Ids: torch.Size([200, 128])
Scores: torch.Size([200, 1])


In [12]:
#splitting the data
indices = list(range(N))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

x_img_train = syn_images[train_idx]
x_img_test = syn_images[test_idx]

x_ids_train = tokens['input_ids'][train_idx]
x_ids_test = tokens['input_ids'][test_idx]

x_mask_train = tokens['attention_mask'][train_idx]
x_mask_test = tokens['attention_mask'][test_idx]

y_train = syn_score[train_idx]
y_test = syn_score[test_idx]

print(f"Train: {len(train_idx)} | Test: {len(test_idx)}")


Train: 160 | Test: 40


In [15]:
#training the data with different learning rates
loss_fn = nn.MSELoss()

#BERT with lower learning rate as it is already pretrained
#Fusion Layer with higher learning rate as it is new and needs training

optimizer = torch.optim.Adam([
    {'params' : model.image_embedder.parameters(), 'lr': 2e-5},
    {'params' : model.text_embedder.parameters(), 'lr': 2e-5},
    {'params' : model.fusion.parameters(), 'lr': 1e-4}
    ])

epochs = 10
train_loss = []
test_loss = []

for epoch in range(epochs):
  model.train()

  preds = model(x_img_train, x_ids_train,x_mask_train)
  loss = loss_fn(preds, y_train)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  #evaluating the model
  model.eval()
  with torch.no_grad():
    test_preds = model(x_img_test, x_ids_test, x_mask_test)
    test_loss = loss_fn(test_preds, y_test)

  train_loss.append(loss.item())
  test_loss.append(test_loss.item())

  print(f"Epoch: {epoch+1:2d} | Train Loss: {loss.item():.3f} | Test Loss: {test_loss.item():.3f}")



AttributeError: 'NoneType' object has no attribute 'size'